# Neural Network from Scratch
**SoC 2026 | Week 2–3**

A fully-connected feedforward neural network — no PyTorch, just NumPy.

**Topics:**
- Forward propagation
- Backpropagation (chain rule)
- Mini-batch gradient descent
- Binary classification on non-linear data (XOR / circles)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(0)

## 1. Dataset — Concentric Circles (not linearly separable)

In [ ]:
X, y = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=0)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

plt.scatter(*X.T, c=y, cmap='bwr', alpha=0.7, edgecolors='k', linewidths=0.3)
plt.title('Circles Dataset'); plt.show()

## 2. MLP Class (2-layer: hidden + output)

In [ ]:
class MLP:
    """
    Architecture: input -> [hidden ReLU] -> [output sigmoid]
    Loss        : Binary cross-entropy
    Optimiser   : Mini-batch SGD
    """
    def __init__(self, n_in, n_hidden, n_out, lr=0.01):
        # He initialisation for ReLU layers
        self.W1 = np.random.randn(n_in, n_hidden) * np.sqrt(2 / n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_out) * np.sqrt(2 / n_hidden)
        self.b2 = np.zeros((1, n_out))
        self.lr = lr

    # ── Activations ──────────────────────────────────────────
    def _relu(self, z):        return np.maximum(0, z)
    def _relu_grad(self, z):   return (z > 0).astype(float)
    def _sigmoid(self, z):     return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    # ── Forward pass ─────────────────────────────────────────
    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self._relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self._sigmoid(self.Z2)
        return self.A2

    # ── Loss ─────────────────────────────────────────────────
    def loss(self, y_hat, y):
        m   = y.shape[0]
        eps = 1e-8
        y   = y.reshape(-1, 1)
        return -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))

    # ── Backward pass ────────────────────────────────────────
    def backward(self, X, y):
        m = X.shape[0]
        y = y.reshape(-1, 1)

        # Output layer gradient
        dZ2 = self.A2 - y                               # dL/dZ2
        dW2 = (self.A1.T @ dZ2) / m
        db2 = np.mean(dZ2, axis=0, keepdims=True)

        # Hidden layer gradient (chain rule through ReLU)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self._relu_grad(self.Z1)
        dW1 = (X.T @ dZ1) / m
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        # Update
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    # ── Train ────────────────────────────────────────────────
    def train(self, X, y, epochs=1000, batch_size=32, verbose=True):
        history = []
        m = X.shape[0]
        for epoch in range(1, epochs + 1):
            # Shuffle
            idx = np.random.permutation(m)
            X_, y_ = X[idx], y[idx]
            for i in range(0, m, batch_size):
                Xb, yb = X_[i:i+batch_size], y_[i:i+batch_size]
                y_hat  = self.forward(Xb)
                self.backward(Xb, yb)
            if epoch % 100 == 0:
                l = self.loss(self.forward(X), y)
                history.append(l)
                if verbose:
                    print(f'Epoch {epoch:4d} | Loss: {l:.4f}')
        return history

    def predict(self, X):
        return (self.forward(X) >= 0.5).astype(int).flatten()

    def accuracy(self, X, y):
        return np.mean(self.predict(X) == y)

## 3. Train & Evaluate

In [ ]:
model = MLP(n_in=2, n_hidden=16, n_out=1, lr=0.05)
history = model.train(X_train, y_train, epochs=1000, batch_size=32)

print(f'\nTrain accuracy : {model.accuracy(X_train, y_train)*100:.2f}%')
print(f'Test  accuracy : {model.accuracy(X_test,  y_test) *100:.2f}%')

## 4. Decision Boundary Visualisation

In [ ]:
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
plt.scatter(*X_test.T, c=y_test, cmap='bwr', edgecolors='k', linewidths=0.4)
plt.title('Decision Boundary — MLP (2 hidden layers)')
plt.show()